# Data Analysis 

In [111]:
## Dependencies
import yfinance as yf
import pandas as pd
import numpy as np

### 1. Acquiring datasets

In [112]:
# NVIDIA dataset:
nvidia = yf.download("NVDA", start="2020-01-01", end="2026-07-01")
nvidia.columns = nvidia.columns.get_level_values(0)
nvidia.columns.name = None # removing title of column names.
nvidia.reset_index(inplace=True) # resetting index to have date as a column.

nvidia["Date"] = pd.to_datetime(nvidia["Date"]) # date column as Date.
nvidia.head(5)

[*********************100%***********************]  1 of 1 completed


,Date,Close,High,Low,Open,Volume
0,2020-01-02,5.963804,5.963804,5.884506,5.934969,237536000
1,2020-01-03,5.868348,5.912099,5.819376,5.844235,205384000
2,2020-01-06,5.892956,5.898176,5.749025,5.775127,262636000
3,2020-01-07,5.964301,6.010041,5.876302,5.921296,314856000
4,2020-01-08,5.975488,6.016753,5.920053,5.960075,277108000


In [113]:
# Semiconductor billing dataset:
sia = pd.read_csv('americas_semiconductor_billings.csv') # one value per month; low granularity compared to nvidia dataset.
sia["Date"] = pd.to_datetime(sia["Date"])
sia.head(5)

/var/folders/7n/53byrv256zx1_3t79kv_fj880000gn/T/ipykernel_94337/2856192465.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  sia["Date"] = pd.to_datetime(sia["Date"])


,Date,Value
0,2026-05-31,46.42M
1,2026-04-30,41.53M
2,2026-03-31,40.60M
3,2026-02-28,36.20M
4,2026-01-31,24.63M


In [114]:
# Merging:
merged = pd.merge(
    nvidia,
    sia[["Date", "Value"]],
    on="Date",
    how="left" # keep the all dates form nvidia set.
)

merged["Value"] = merged["Value"].fillna("")

## Output:
display(merged[merged["Value"] != ""].head(5)) # value column filled only at end of month.
display(merged.head(10))

,Date,Close,High,Low,Open,Volume,Value
607,2022-05-31,18.616373,19.142800,18.295332,18.923454,664100000,12.61M
628,2022-06-30,15.117033,15.523903,14.820855,15.318473,686070000,12.10M
671,2022-08-31,15.052211,15.496976,14.917584,15.341408,573710000,11.52M
692,2022-09-30,12.108991,12.601770,12.045149,12.057119,565638000,13.77M
713,2022-10-31,13.463633,13.803790,13.264127,13.743938,486341000,11.69M


,Date,Close,High,Low,Open,Volume,Value
0,2020-01-02,5.963804,5.963804,5.884506,5.934969,237536000,
1,2020-01-03,5.868348,5.912099,5.819376,5.844235,205384000,
2,2020-01-06,5.892956,5.898176,5.749025,5.775127,262636000,
3,2020-01-07,5.964301,6.010041,5.876302,5.921296,314856000,
4,2020-01-08,5.975488,6.016753,5.920053,5.960075,277108000,
5,2020-01-09,6.041114,6.113452,5.987420,6.061746,255112000,
6,2020-01-10,6.073430,6.178582,6.059261,6.148254,316296000,
7,2020-01-13,6.263847,6.288954,6.133837,6.156459,319840000,
8,2020-01-14,6.147011,6.246445,6.133835,6.221089,359088000,
9,2020-01-15,6.104502,6.182061,6.078649,6.159688,263104000,


### 2. Data Inspection

In [115]:
# Inspecting and cleaning time series data

## Date as index in both datasets:
sia.set_index("Date", inplace=True)
nvidia.set_index("Date", inplace=True)


In [116]:
## Duplicates
print(sia.duplicated().sum()) # 1 potential duplicate in sia.
print(nvidia.duplicated().sum())

1
0


In [117]:
sia[sia["Value"].duplicated(keep=False)] # duplicate is just the value in different months; duplicate should NOT be removed.

,Value
Date,
2025-03-31,20.71M
2024-11-30,20.71M


In [118]:
## Null
nvidia.isnull().sum()
sia.isnull().sum()
# NONE

Value    0
dtype: int64

In [119]:
## Differences in time stamps (now, index)
nvidia.index.to_series().diff().value_counts()

Date
1 days    1276
3 days     291
4 days      48
2 days      15
Name: count, dtype: int64

In [120]:
## month-end data from nividia:
nvidia_monthly = nvidia.resample("ME").last()
nvidia_monthly.head(5)

,Close,High,Low,Open,Volume
Date,,,,,
2020-01-31,5.877296,6.076661,5.835534,6.064729,370420000
2020-02-29,6.717552,6.777000,6.014133,6.030798,1133252000
2020-03-31,6.556621,6.850126,6.411111,6.646164,949960000
2020-04-30,7.269990,7.423708,7.256061,7.369732,375916000
2020-05-31,8.830544,8.830544,8.442022,8.511170,745256000


### 2a. Feature Engineering

In [121]:
# turning Value from string to numeric:
sia["Value_num"] = sia["Value"].str.replace("M", "", regex=False)

sia["Value_num"] = pd.to_numeric(sia["Value_num"])

In [122]:
# Lagging by one day:
nvidia["Close_lag1"] = nvidia["Close"].shift(1)
nvidia["Close_lag2"] = nvidia["Close"].shift(2)

sia["Value_lag1"] = sia["Value_num"].shift(1)
sia["Value_lag2"] = sia["Value_num"].shift(2)

In [123]:
# Rolling statistics: helps capture short-term trends
## Rolling averages, e.g.: how mean sales change over a rolling 5-day period?

nvidia["rolling_mean_3"] = nvidia["Close"].rolling(3).mean()
nvidia["rolling_sd_3"] = nvidia["Close"].rolling(3).std()

sia["rolling_mean_3"] = sia["Value_num"].rolling(3).mean()
sia["rolling_sd_3"] = sia["Value_num"].rolling(3).std()

In [124]:
# Expanding Window features:
## For 'cumulative' information
nvidia["expanding_mean"] = nvidia["Close"].expanding().mean() # cumulative/'expanding' mean per row, i.e. mean of row 1, mean of rows 1 and 2, and so on.
sia["expanding_mean"] = sia["Value_num"].expanding().mean()

In [125]:
# % change feature:
nvidia["return"] = nvidia["Close"].pct_change() # pct(%) change in Close value from previous row (lag of 1 is default)
sia["Value_growth"] = sia["Value_num"].pct_change()

In [126]:
# Date component extraction:
nvidia["Year"] = nvidia.index.year
nvidia["Month"] = nvidia.index.month

sia["Year"] = sia.index.year
sia["Month"] = sia.index.month

### 2b. Merging Multi-Source Data

In [127]:
# Merging datasets:

## a common column that captures the year and month (Y-m), by capturing the monthly frequency, i.e. to_period("M")
nvidia["YM"] = nvidia.index.to_period("M")
sia["YM"] = sia.index.to_period("M")

final = pd.merge(nvidia.reset_index(), sia.reset_index(), # turns index (or Date) into a dataset column => required for merging
                 on="YM",
                 how="inner") # keeps the months that appear in both datasets in 'final'

In [133]:
final.head()

,Date_x,Close,High,Low,Open,Volume,Close_lag1,Close_lag2,rolling_mean_3_x,rolling_sd_3_x,...,Value,Value_num,Value_lag1,Value_lag2,rolling_mean_3_y,rolling_sd_3_y,expanding_mean_y,Value_growth,Year_y,Month_y
0,2022-04-01,26.632416,27.414080,26.188742,27.293441,517235000,27.204706,27.607504,27.148209,0.489993,...,11.63M,11.63,12.61,12.1,12.113333,0.490136,17.6402,-0.077716,2022,4
1,2022-04-04,27.278482,27.475892,26.533708,26.648365,397120000,26.632416,27.204706,27.038535,0.353639,...,11.63M,11.63,12.61,12.1,12.113333,0.490136,17.6402,-0.077716,2022,4
2,2022-04-05,25.853745,27.237610,25.743075,27.172803,436615000,27.278482,26.632416,26.588214,0.713397,...,11.63M,11.63,12.61,12.1,12.113333,0.490136,17.6402,-0.077716,2022,4
3,2022-04-06,24.334280,25.224619,23.931484,24.859710,703833000,25.853745,27.278482,25.822169,1.472355,...,11.63M,11.63,12.61,12.1,12.113333,0.490136,17.6402,-0.077716,2022,4
4,2022-04-07,24.135874,24.648342,23.408049,24.368179,557992000,24.334280,25.853745,24.774633,0.939789,...,11.63M,11.63,12.61,12.1,12.113333,0.490136,17.6402,-0.077716,2022,4
